# 06. Mann–Whitney U Test

Phân tích thống kê sự khác biệt về **phân bố feature giữa hai nhóm `churn_180`**.

### Câu hỏi

> Những đặc trưng hành vi nào có phân bố khác nhau giữa nhóm `churn_180 = 0` và `churn_180 = 1`, và mức độ khác biệt lớn đến đâu?

### Nguyên tắc

- `churn_180` chỉ là **biến nhóm**, không phải feature kiểm định.
- Mỗi feature được kiểm định riêng giữa hai nhóm churn.
- Dùng **Mann–Whitney U** vì không yêu cầu feature phải tuân theo phân phối chuẩn.
- Báo cáo đồng thời **median, U statistic, p-value, p-value sau FDR và effect size**.
- Không diễn giải kết quả là quan hệ nhân quả.


## 01. Load Feature Dataset

Đọc trực tiếp output của Feature Engineering.

Không đọc lại các CSV gốc và không thay đổi các feature đã tạo.


In [4]:
!pip install -q hvplot scipy statsmodels

import pandas as pd
import numpy as np
import hvplot.pandas

from google.colab import drive
from pyspark.sql import SparkSession, functions as F
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

drive.mount("/content/drive", force_remount=True)

DATA_PATH = (
    "/content/drive/MyDrive/BigData/DA-BigData/code/data/model_input/"
    "customer_churn_model_dataset.csv"
)

spark = (
    SparkSession.builder
    .appName("Olist-Mann-Whitney-Analysis")
    .getOrCreate()
)

df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(DATA_PATH)
)

print("Using:", DATA_PATH)
print("Rows:", df.count())
print("Columns:", len(df.columns))


Mounted at /content/drive
Using: /content/drive/MyDrive/BigData/DA-BigData/code/data/model_input/customer_churn_model_dataset.csv
Rows: 213742
Columns: 36


## 02. Define Analysis Population

Đơn vị thống kê cần được xác định trước khi kiểm định.

Dataset churn có nhiều `snapshot_date` cho một `customer_unique_id`. Nếu dùng tất cả snapshot như các quan sát độc lập, cùng một khách hàng có thể xuất hiện nhiều lần và làm sai giả định độc lập của Mann–Whitney.

Vì vậy, notebook sử dụng **snapshot gần nhất của mỗi customer** để có:

> **1 customer = 1 statistical observation**

Nếu `eligible_180` tồn tại, chỉ giữ các observation có đủ điều kiện quan sát nhãn 180 ngày.


In [5]:
# Giữ snapshot gần nhất của mỗi customer.
# Đây chỉ là cách xác định population cho kiểm định thống kê,
# không thay đổi dataset Feature Engineering gốc.

if "snapshot_date" not in df.columns:
    raise ValueError("Dataset cần có snapshot_date.")

if "customer_unique_id" not in df.columns:
    raise ValueError("Dataset cần có customer_unique_id.")

customer_window = (
    df
    .withColumn(
        "_snapshot_ts",
        F.to_timestamp("snapshot_date")
    )
)

from pyspark.sql.window import Window

latest_window = (
    Window
    .partitionBy("customer_unique_id")
    .orderBy(F.col("_snapshot_ts").desc())
)

analysis_df = (
    customer_window
    .withColumn("_rn", F.row_number().over(latest_window))
    .filter(F.col("_rn") == 1)
    .drop("_rn", "_snapshot_ts")
)

# Chỉ dùng nhãn 180 ngày có đủ điều kiện quan sát nếu cột này tồn tại.
if "eligible_180" in analysis_df.columns:
    analysis_df = analysis_df.filter(
        F.col("eligible_180") == 1
    )

analysis_df = analysis_df.filter(
    F.col("churn_180").isNotNull()
)

print("Analysis rows:", analysis_df.count())

analysis_df.groupBy("churn_180").count().orderBy("churn_180").show()


Analysis rows: 51998
+---------+-----+
|churn_180|count|
+---------+-----+
|        0|  511|
|        1|51487|
+---------+-----+



## 03. Define Test Features

Kế thừa bộ feature đã được làm sạch ở bước Spearman.

Không đưa vào kiểm định:
- `churn_180`: biến nhóm
- `customer_unique_id`: định danh
- `snapshot_date`: thời gian snapshot

Các feature có quan hệ dẫn xuất trực tiếp đã được loại ở bước Spearman cũng không đưa vào đây để giữ nhất quán giữa hai phân tích.


In [8]:
RFM_FEATURES = [
    "recency_days",
    "frequency_orders",
    "monetary_matched_total"
]

PURCHASE_FEATURES = [
    "orders_last_30d",
    "orders_last_90d",
    "orders_last_180d"
]

DELIVERY_FEATURES = [
    "avg_delivery_days",
    "late_delivery_ratio"
]

REVIEW_FEATURES = [
    "review_count",
    "avg_review_score"
]

PRODUCT_FEATURES = [
    "total_items",
    "unique_categories",
    "avg_product_price"
]

FEATURE_GROUPS = {
    "RFM": RFM_FEATURES,
    "Purchase Behavior": PURCHASE_FEATURES,
    "Delivery": DELIVERY_FEATURES,
    "Review": REVIEW_FEATURES,
    "Product": PRODUCT_FEATURES
}

TEST_FEATURES = [
    feature
    for group in FEATURE_GROUPS.values()
    for feature in group
]

missing_features = sorted(
    set(TEST_FEATURES) - set(analysis_df.columns)
)

if missing_features:
    raise ValueError(
        f"Thiếu feature trong dataset: {missing_features}"
    )

print("Number of test features:", len(TEST_FEATURES))
print(TEST_FEATURES)


Number of test features: 13
['recency_days', 'frequency_orders', 'monetary_matched_total', 'orders_last_30d', 'orders_last_90d', 'orders_last_180d', 'avg_delivery_days', 'late_delivery_ratio', 'review_count', 'avg_review_score', 'total_items', 'unique_categories', 'avg_product_price']


## 04. Prepare Two Churn Groups

Mỗi feature sẽ được tách thành hai mẫu:

- `churn_180 = 0`
- `churn_180 = 1`

Sau đó Mann–Whitney kiểm tra xem hai mẫu có khác nhau về phân bố/rank hay không.


In [9]:
churn_0 = (
    analysis_df
    .filter(F.col("churn_180") == 0)
    .count()
)

churn_1 = (
    analysis_df
    .filter(F.col("churn_180") == 1)
    .count()
)

print(f"Churn = 0: {churn_0:,}")
print(f"Churn = 1: {churn_1:,}")

if churn_0 == 0 or churn_1 == 0:
    raise ValueError(
        "Mann–Whitney cần cả hai nhóm churn_180 = 0 và churn_180 = 1."
    )


Churn = 0: 511
Churn = 1: 51,487


## 05. Mann–Whitney U Test

### Giả thuyết

**H₀:** Phân bố/rank của feature giữa hai nhóm `churn_180 = 0` và `churn_180 = 1` không khác nhau.

**H₁:** Hai nhóm có sự khác biệt.

Kiểm định được thực hiện hai phía (`two-sided`).

Ngoài U và p-value, tính thêm:
- Median của mỗi nhóm
- IQR của mỗi nhóm
- Rank-biserial effect size


In [10]:
def iqr(values):
    return np.percentile(values, 75) - np.percentile(values, 25)


def rank_biserial_from_u(u_stat, n_churn_1, n_churn_0):
    # U được tính với churn_1 là sample thứ nhất.
    # Giá trị dương: feature có xu hướng cao hơn ở churn_1.
    return (
        2 * u_stat / (n_churn_1 * n_churn_0)
    ) - 1


def mann_whitney_test(data, feature):
    values = (
        data
        .select(
            F.col("churn_180").cast("int"),
            F.col(feature).cast("double")
        )
        .na.drop()
        .toPandas()
    )

    group_0 = values.loc[
        values["churn_180"] == 0,
        feature
    ].to_numpy()

    group_1 = values.loc[
        values["churn_180"] == 1,
        feature
    ].to_numpy()

    if len(group_0) == 0 or len(group_1) == 0:
        return None

    u_stat, p_value = mannwhitneyu(
        group_1,
        group_0,
        alternative="two-sided",
        method="asymptotic"
    )

    return {
        "feature": feature,
        "n_churn_0": len(group_0),
        "n_churn_1": len(group_1),
        "median_churn_0": np.median(group_0),
        "median_churn_1": np.median(group_1),
        "iqr_churn_0": iqr(group_0),
        "iqr_churn_1": iqr(group_1),
        "u_statistic": float(u_stat),
        "p_value": float(p_value),
        "rank_biserial": rank_biserial_from_u(
            u_stat,
            len(group_1),
            len(group_0)
        )
    }


## 06. Run Tests for All Features


In [12]:
results = []

for group_name, features in FEATURE_GROUPS.items():
    for feature in features:

        result = mann_whitney_test(
            analysis_df,
            feature
        )

        if result is not None:
            result["group"] = group_name
            results.append(result)

mw_results = (
    pd.DataFrame(results)
    .loc[
        :,
        [
            "group",
            "feature",
            "n_churn_0",
            "n_churn_1",
            "median_churn_0",
            "median_churn_1",
            "iqr_churn_0",
            "iqr_churn_1",
            "u_statistic",
            "p_value",
            "rank_biserial"
        ]
    ]
)

mw_results


,group,feature,n_churn_0,n_churn_1,median_churn_0,median_churn_1,iqr_churn_0,iqr_churn_1,u_statistic,p_value,rank_biserial
0,RFM,recency_days,511,51487,101.09831,146.691921,108.649815,83.928547,15782460.0,7.148296e-15,0.199737
1,RFM,frequency_orders,511,51487,1.00000,1.000000,0.000000,0.000000,12066675.5,3.564773e-29,-0.082726
2,RFM,monetary_matched_total,506,51151,112.66000,105.380000,132.027500,116.050000,12589352.5,2.918423e-01,-0.027188
3,Purchase Behavior,orders_last_30d,511,51487,0.00000,0.000000,0.000000,0.000000,12461832.0,2.213065e-06,-0.052687
4,Purchase Behavior,orders_last_90d,511,51487,0.00000,0.000000,1.000000,1.000000,11257171.0,1.049632e-12,-0.144262
5,Purchase Behavior,orders_last_180d,511,51487,1.00000,1.000000,0.000000,0.000000,12624170.0,3.121301e-11,-0.040347
6,Delivery,avg_delivery_days,511,51466,10.10603,11.082734,8.010168,8.696536,14169685.5,2.507192e-03,0.077578
7,Delivery,late_delivery_ratio,511,51487,0.00000,0.000000,0.000000,0.000000,13171591.0,8.975457e-01,0.001267
8,Review,review_count,511,51487,1.00000,1.000000,0.000000,0.000000,12339969.0,1.462703e-10,-0.061951
9,Review,avg_review_score,488,50006,5.00000,5.000000,1.000000,1.000000,11643340.0,4.946434e-02,-0.045742


## 07. Multiple Testing Correction

Có nhiều feature được kiểm định cùng lúc. Vì vậy p-value được điều chỉnh bằng **Benjamini–Hochberg FDR**.

`p_value_fdr` là p-value sau hiệu chỉnh.

Trong phần diễn giải, dùng `p_value_fdr` thay vì chỉ dùng p-value gốc.


In [14]:
reject, p_value_fdr, _, _ = multipletests(
    mw_results["p_value"],
    alpha=0.05,
    method="fdr_bh"
)

mw_results["p_value_fdr"] = p_value_fdr
mw_results["significant_fdr_0_05"] = reject

mw_results = (
    mw_results
    .sort_values(
        ["p_value_fdr", "p_value"],
        ascending=True
    )
    .reset_index(drop=True)
)

mw_results


,group,feature,n_churn_0,n_churn_1,median_churn_0,median_churn_1,iqr_churn_0,iqr_churn_1,u_statistic,p_value,rank_biserial,p_value_fdr,significant_fdr_0_05
0,RFM,frequency_orders,511,51487,1.00000,1.000000,0.000000,0.000000,12066675.5,3.564773e-29,-0.082726,4.634205e-28,True
1,RFM,recency_days,511,51487,101.09831,146.691921,108.649815,83.928547,15782460.0,7.148296e-15,0.199737,4.646392e-14,True
2,Product,total_items,511,51487,1.00000,1.000000,0.000000,0.000000,11712759.0,7.208896e-14,-0.109630,3.123855e-13,True
3,Purchase Behavior,orders_last_90d,511,51487,0.00000,0.000000,1.000000,1.000000,11257171.0,1.049632e-12,-0.144262,3.411304e-12,True
4,Purchase Behavior,orders_last_180d,511,51487,1.00000,1.000000,0.000000,0.000000,12624170.0,3.121301e-11,-0.040347,8.115383e-11,True
5,Review,review_count,511,51487,1.00000,1.000000,0.000000,0.000000,12339969.0,1.462703e-10,-0.061951,3.169190e-10,True
6,Product,unique_categories,511,51487,1.00000,1.000000,0.000000,0.000000,12520437.5,1.494562e-08,-0.048232,2.775615e-08,True
7,Purchase Behavior,orders_last_30d,511,51487,0.00000,0.000000,0.000000,0.000000,12461832.0,2.213065e-06,-0.052687,3.596230e-06,True
8,Delivery,avg_delivery_days,511,51466,10.10603,11.082734,8.010168,8.696536,14169685.5,2.507192e-03,0.077578,3.621500e-03,True
9,Product,avg_product_price,511,51487,70.74000,78.945000,91.045000,97.820000,13819002.5,4.920514e-02,0.050481,5.845786e-02,False


## 08. Interpret Effect Size

`rank_biserial` cho biết **hướng và mức độ khác biệt về thứ hạng** giữa hai nhóm.

- Dương: feature có xu hướng cao hơn ở `churn_180 = 1`.
- Âm: feature có xu hướng thấp hơn ở `churn_180 = 1`.
- Gần 0: hai nhóm có mức độ chồng lấn lớn hơn.

Effect size được xem cùng median và p-value FDR, không dùng p-value đơn độc để kết luận.


In [15]:
display(
    mw_results[
        [
            "group",
            "feature",
            "median_churn_0",
            "median_churn_1",
            "u_statistic",
            "p_value",
            "p_value_fdr",
            "rank_biserial",
            "significant_fdr_0_05"
        ]
    ]
)


,group,feature,median_churn_0,median_churn_1,u_statistic,p_value,p_value_fdr,rank_biserial,significant_fdr_0_05
0,RFM,frequency_orders,1.00000,1.000000,12066675.5,3.564773e-29,4.634205e-28,-0.082726,True
1,RFM,recency_days,101.09831,146.691921,15782460.0,7.148296e-15,4.646392e-14,0.199737,True
2,Product,total_items,1.00000,1.000000,11712759.0,7.208896e-14,3.123855e-13,-0.109630,True
3,Purchase Behavior,orders_last_90d,0.00000,0.000000,11257171.0,1.049632e-12,3.411304e-12,-0.144262,True
4,Purchase Behavior,orders_last_180d,1.00000,1.000000,12624170.0,3.121301e-11,8.115383e-11,-0.040347,True
5,Review,review_count,1.00000,1.000000,12339969.0,1.462703e-10,3.169190e-10,-0.061951,True
6,Product,unique_categories,1.00000,1.000000,12520437.5,1.494562e-08,2.775615e-08,-0.048232,True
7,Purchase Behavior,orders_last_30d,0.00000,0.000000,12461832.0,2.213065e-06,3.596230e-06,-0.052687,True
8,Delivery,avg_delivery_days,10.10603,11.082734,14169685.5,2.507192e-03,3.621500e-03,0.077578,True
9,Product,avg_product_price,70.74000,78.945000,13819002.5,4.920514e-02,5.845786e-02,0.050481,False


## 09. Visualization

Dùng **hvPlot** để:

1. So sánh effect size của toàn bộ feature.
2. Vẽ boxplot cho các feature có bằng chứng khác biệt sau FDR.

Không gom các feature khác đơn vị vào cùng một boxplot raw-scale.


In [20]:
import pandas as pd
import hvplot.pandas


# Copy kết quả Mann–Whitney
effect_plot = mw_results.copy()


# Đảm bảo effect size là kiểu số
effect_plot["rank_biserial"] = pd.to_numeric(
    effect_plot["rank_biserial"],
    errors="coerce"
)


# Loại các dòng không có effect size hợp lệ
effect_plot = effect_plot.dropna(
    subset=["rank_biserial"]
)


# Tạo nhãn hiển thị
effect_plot["feature_label"] = (
    effect_plot["group"].astype(str)
    + " | "
    + effect_plot["feature"].astype(str)
)


# Sắp xếp theo effect size
effect_plot = effect_plot.sort_values(
    "rank_biserial"
)


# Visualization
effect_plot.hvplot.bar(
    x="feature_label",
    y="rank_biserial",
    width=1100,
    height=600,
    title="Mann–Whitney Rank-Biserial Effect Size",
    xlabel="Feature",
    ylabel="Rank-Biserial Effect Size",
    rot=60,
    ylim=(
        effect_plot["rank_biserial"].min() - 0.03,
        effect_plot["rank_biserial"].max() + 0.03
    ),
    hover_cols=[
        "group",
        "feature",
        "rank_biserial",
        "p_value",
        "p_value_fdr"
    ],
    grid=True
)

:Bars   [feature_label]   (rank_biserial,group,feature,p_value,p_value_fdr)

In [21]:
significant_features = (
    mw_results.loc[
        mw_results["significant_fdr_0_05"],
        "feature"
    ]
    .tolist()
)

print("Features significant after FDR:", significant_features)


Features significant after FDR: ['frequency_orders', 'recency_days', 'total_items', 'orders_last_90d', 'orders_last_180d', 'review_count', 'unique_categories', 'orders_last_30d', 'avg_delivery_days']


In [22]:
# Chỉ vẽ boxplot cho các feature có bằng chứng khác biệt sau FDR.
# Nếu không có feature nào, notebook không cố tạo biểu đồ giả.

plots = []

for feature in significant_features:

    plot_df = (
        analysis_df
        .select(
            "churn_180",
            feature
        )
        .withColumn(
            feature,
            F.col(feature).cast("double")
        )
        .na.drop()
        .toPandas()
    )

    plot_df["churn_group"] = (
        "Churn = "
        + plot_df["churn_180"].astype(str)
    )

    plots.append(
        plot_df.hvplot.box(
            y=feature,
            by="churn_group",
            width=500,
            height=400,
            title=f"{feature}: Distribution by Churn Group",
            ylabel=feature,
            xlabel="Churn Group",
            tools=["hover"]
        )
    )

if plots:
    display(
        plots[0]
        if len(plots) == 1
        else __import__("holoviews").Layout(plots).cols(2)
    )
else:
    print(
        "Không có feature nào đạt FDR < 0.05; "
        "không vẽ boxplot nhóm."
    )


:Layout
   .BoxWhisker.I    :BoxWhisker   [churn_group]   (frequency_orders)
   .BoxWhisker.II   :BoxWhisker   [churn_group]   (recency_days)
   .BoxWhisker.III  :BoxWhisker   [churn_group]   (total_items)
   .BoxWhisker.IV   :BoxWhisker   [churn_group]   (orders_last_90d)
   .BoxWhisker.V    :BoxWhisker   [churn_group]   (orders_last_180d)
   .BoxWhisker.VI   :BoxWhisker   [churn_group]   (review_count)
   .BoxWhisker.VII  :BoxWhisker   [churn_group]   (unique_categories)
   .BoxWhisker.VIII :BoxWhisker   [churn_group]   (orders_last_30d)
   .BoxWhisker.IX   :BoxWhisker   [churn_group]   (avg_delivery_days)

## 10. Take Notes

Mann–Whitney được dùng ở đây để kiểm tra **sự khác biệt về phân bố feature giữa hai nhóm churn**, không phải để chứng minh quan hệ nhân quả.

Khi kết luận cần xem đồng thời:

- Median của `churn_180 = 0` và `churn_180 = 1`
- `p_value_fdr`
- Rank-biserial effect size
- Phân bố qua visualization

Nếu một feature có `p_value_fdr < 0.05`, có bằng chứng thống kê rằng phân bố feature giữa hai nhóm khác nhau trong population được phân tích. Mức độ khác biệt được đánh giá thêm bằng effect size.

**Không kết luận rằng feature gây ra churn.**

Kết quả của notebook này là một phân tích thống kê mô tả sự khác biệt giữa hai nhóm, bổ sung cho Spearman và không thay thế mô hình Machine Learning.


In [ ]:
# Bảng cuối cùng để dùng khi viết báo cáo

final_mw_summary = (
    mw_results[
        [
            "group",
            "feature",
            "n_churn_0",
            "n_churn_1",
            "median_churn_0",
            "median_churn_1",
            "u_statistic",
            "p_value",
            "p_value_fdr",
            "rank_biserial",
            "significant_fdr_0_05"
        ]
    ]
)

display(final_mw_summary)

OUTPUT_PATH = (
    "/content/drive/MyDrive/BigData/DA-BigData/code/data/model_input/"
    "mann_whitney_results.csv"
)

final_mw_summary.to_csv(
    OUTPUT_PATH,
    index=False
)

print("Saved:", OUTPUT_PATH)
